# Silver play-by-play

## Pipeline overview

**Purpose:** Preserve the nflverse play-level record in a clean, reusable Silver table. This notebook does not classify eligible plays, aggregate team metrics, or create fantasy features. It is a new output alongside the existing standardized-plays pipeline.

| Role | Files | Columns | Grain |
| --- | --- | ---: | --- |
| Primary input | `data/bronze/pbp/pbp_{season}.parquet` | 372 currently | `game_id + play_id` |
| Validation-only input | `data/bronze/schedules/schedules_{season}.parquet` | 46 currently | `game_id` |
| Output | `data/silver/pbp/silver_pbp_{season}.parquet` | 372, matching PBP input | `game_id + play_id` |

**Seasons:** 2023–2026. The 2026 season is in progress; row counts are intentionally not hard-coded. The output retains every source column and play type, including rows with no possession team.

### Cleaning and transformations

1. Require the same named PBP columns in every season partition. Preserve source column order.
2. Count and remove only exact full-row duplicates. Never select arbitrarily among conflicting rows with the same play key.
3. Convert integral-valued `play_id`, `drive`, drive-play references, and selected count fields to nullable integers. Keep `game_id` and player IDs as strings, `epa` as a float, and null values as null. Convert the binary `success` indicator to nullable integer 0/1.
4. Parse `game_date` as a datetime. No rows or columns are added through the schedule validation join. Other source fields keep their existing values and types until their semantics justify a specific conversion.

### Validation and quality checks

- Required files, columns, and source schema shape; safe integer and date parsing.
- Exact-duplicate count and `Silver rows = Bronze rows - exact duplicates`; non-null, unique `game_id + play_id` after conversion.
- Every play matches exactly one schedule game; season, week, season type, date, and populated offense/defense teams agree with that game.
- Source values and nulls reconcile except approved type conversions and exact-row deduplication; `epa` and other metrics are never filled.
- Output has the same named columns, expected nullable types, and an equal Parquet round trip.


## 1. Set paths and the type contract

All transformations remain visible here while this Silver contract is being validated. `drive_play_id_started` and `drive_play_id_ended` refer to `play_id`, so they use the same numeric type. `success` is a source 0/1 indicator; it is not recalculated from EPA.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


SEASONS = [2023, 2024, 2025, 2026]
PLAY_KEY = ["game_id", "play_id"]
EXPECTED_PBP_COLUMNS = 372
EXPECTED_SCHEDULE_COLUMNS = 46
POSTSEASON_GAME_TYPES = {"WC", "DIV", "CON", "SB"}

INTEGER_COLUMNS = [
    "play_id",
    "drive",
    "drive_play_id_started",
    "drive_play_id_ended",
    "down",
    "ydstogo",
    "yards_gained",
    "posteam_score",
    "defteam_score",
]

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ROADMAP.md").exists():
    PROJECT_ROOT = next(
        parent for parent in PROJECT_ROOT.parents
        if (parent / "ROADMAP.md").exists()
    )

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
SILVER_PBP_DIR = PROJECT_ROOT / "data" / "silver" / "pbp"

print(f"Project root: {PROJECT_ROOT}")
print(f"Seasons: {SEASONS}")

Project root: /Users/dtwice/Development/dev_sports/nfl/fantasy-football/nfl-dream-lab-app
Seasons: [2023, 2024, 2025, 2026]


## 2. Inspect source files and schema

Column-name drift is a stop condition. All-null source columns can legitimately have different Parquet physical types across seasons, so this check compares names and order rather than requiring every Arrow type to match.

In [2]:
import pyarrow.parquet as pq


source_columns = None
source_inventory = []

for season in SEASONS:
    pbp_path = BRONZE_DIR / "pbp" / f"pbp_{season}.parquet"
    schedule_path = BRONZE_DIR / "schedules" / f"schedules_{season}.parquet"
    assert pbp_path.exists(), f"Missing PBP: {pbp_path}"
    assert schedule_path.exists(), f"Missing schedule: {schedule_path}"

    pbp_schema = pq.read_schema(pbp_path)
    schedule_schema = pq.read_schema(schedule_path)
    assert len(pbp_schema) == EXPECTED_PBP_COLUMNS, (season, len(pbp_schema))
    assert len(schedule_schema) == EXPECTED_SCHEDULE_COLUMNS, (season, len(schedule_schema))
    assert set(PLAY_KEY + INTEGER_COLUMNS + ["game_date", "success", "epa", "season", "week", "season_type", "posteam", "defteam"]).issubset(pbp_schema.names)
    assert {"game_id", "season", "week", "game_type", "gameday", "home_team", "away_team"}.issubset(schedule_schema.names)

    if source_columns is None:
        source_columns = pbp_schema.names
    else:
        assert pbp_schema.names == source_columns, f"PBP column drift in {season}"

    source_inventory.append(
        {
            "season": season,
            "pbp_rows": pq.ParquetFile(pbp_path).metadata.num_rows,
            "pbp_columns": len(pbp_schema),
            "schedule_rows": pq.ParquetFile(schedule_path).metadata.num_rows,
            "schedule_columns": len(schedule_schema),
        }
    )

pd.DataFrame(source_inventory)

,season,pbp_rows,pbp_columns,schedule_rows,schedule_columns
0,2023,49665,372,285,46
1,2024,49492,372,285,46
2,2025,48771,372,285,46
3,2026,14031,372,272,46


## 3. Clean each season and validate its play grain

The full-row duplicate check runs before type normalization. A repeated play key with different facts is a conflict, not a duplicate to discard. Integer conversions reject fractional, non-finite, and unsafe-magnitude values instead of silently rounding them.

In [3]:
validation_results = []
SILVER_PBP_DIR.mkdir(parents=True, exist_ok=True)

for season in SEASONS:
    pbp_path = BRONZE_DIR / "pbp" / f"pbp_{season}.parquet"
    schedule_path = BRONZE_DIR / "schedules" / f"schedules_{season}.parquet"
    output_path = SILVER_PBP_DIR / f"silver_pbp_{season}.parquet"

    bronze_plays = pd.read_parquet(pbp_path)
    assert bronze_plays.columns.tolist() == source_columns
    assert bronze_plays["season"].eq(season).all()

    exact_duplicate_mask = bronze_plays.duplicated(keep="first")
    exact_duplicate_count = int(exact_duplicate_mask.sum())
    source_without_duplicates = bronze_plays.loc[~exact_duplicate_mask].reset_index(drop=True)
    silver_plays = source_without_duplicates.copy()
    assert len(silver_plays) == len(bronze_plays) - exact_duplicate_count

    for column in INTEGER_COLUMNS + ["success"]:
        numeric_values = pd.to_numeric(silver_plays[column], errors="raise")
        populated_values = numeric_values.dropna()
        assert np.isfinite(populated_values).all(), (season, column, "non-finite")
        assert populated_values.mod(1).eq(0).all(), (season, column, "fractional")
        assert populated_values.abs().le(2**53).all(), (season, column, "unsafe float integer")
        if column == "success":
            assert populated_values.isin([0, 1]).all(), (season, column, "not 0/1")
            silver_plays[column] = numeric_values.astype("Int8")
        else:
            silver_plays[column] = numeric_values.astype("Int64")

    silver_plays["game_date"] = pd.to_datetime(
        silver_plays["game_date"], format="%Y-%m-%d", errors="raise"
    )
    assert silver_plays["game_date"].notna().all()
    assert silver_plays[PLAY_KEY].notna().all().all()
    assert not silver_plays.duplicated(PLAY_KEY).any(), (season, "conflicting play keys")
    assert silver_plays["game_id"].map(type).eq(str).all()
    assert pd.api.types.is_float_dtype(silver_plays["epa"])

    # Compare untouched fields, converted numbers, and the parsed date to Bronze.
    changed_columns = set(INTEGER_COLUMNS + ["success", "game_date"])
    untouched_columns = [name for name in source_columns if name not in changed_columns]
    pd.testing.assert_frame_equal(
        source_without_duplicates[untouched_columns],
        silver_plays[untouched_columns],
    )
    for column in INTEGER_COLUMNS + ["success"]:
        pd.testing.assert_series_equal(
            pd.to_numeric(source_without_duplicates[column]).astype("Float64"),
            silver_plays[column].astype("Float64"),
            check_names=False,
        )
    assert silver_plays["game_date"].dt.strftime("%Y-%m-%d").equals(
        source_without_duplicates["game_date"]
    )

    # Schedules provide validation context only; none of their columns enter Silver.
    schedules = pd.read_parquet(
        schedule_path,
        columns=["game_id", "season", "week", "game_type", "gameday", "home_team", "away_team"],
    )
    assert schedules["game_id"].notna().all()
    assert not schedules.duplicated("game_id").any()
    schedule_check = silver_plays[
        ["game_id", "season", "week", "season_type", "game_date", "posteam", "defteam"]
    ].merge(
        schedules, on="game_id", how="left", validate="many_to_one",
        indicator=True, suffixes=("_pbp", "_schedule"),
    )
    assert len(schedule_check) == len(silver_plays)
    assert schedule_check["_merge"].eq("both").all()
    assert schedule_check["season_pbp"].eq(schedule_check["season_schedule"]).all()
    assert schedule_check["week_pbp"].eq(schedule_check["week_schedule"]).all()
    expected_season_type = schedule_check["game_type"].replace(
        {game_type: "POST" for game_type in POSTSEASON_GAME_TYPES}
    )
    assert schedule_check["season_type"].eq(expected_season_type).all()
    assert schedule_check["game_date"].eq(
        pd.to_datetime(schedule_check["gameday"], format="%Y-%m-%d", errors="raise")
    ).all()
    for team_column in ["posteam", "defteam"]:
        populated = schedule_check[team_column].notna()
        assert (
            schedule_check.loc[populated, team_column].eq(schedule_check.loc[populated, "home_team"])
            | schedule_check.loc[populated, team_column].eq(schedule_check.loc[populated, "away_team"])
        ).all(), (season, team_column)
    both_teams_present = schedule_check[["posteam", "defteam"]].notna().all(axis=1)
    assert schedule_check.loc[both_teams_present, "posteam"].ne(
        schedule_check.loc[both_teams_present, "defteam"]
    ).all()

    assert silver_plays.columns.tolist() == source_columns
    silver_plays.to_parquet(output_path, index=False)
    reloaded = pd.read_parquet(output_path)
    pd.testing.assert_frame_equal(reloaded, silver_plays)
    assert str(reloaded["play_id"].dtype) == "Int64"
    assert str(reloaded["success"].dtype) == "Int8"
    assert pd.api.types.is_datetime64_any_dtype(reloaded["game_date"])

    validation_results.append(
        {
            "season": season,
            "bronze_rows": len(bronze_plays),
            "exact_duplicates_removed": exact_duplicate_count,
            "silver_rows": len(silver_plays),
            "columns": len(silver_plays.columns),
            "games": silver_plays["game_id"].nunique(),
            "output": str(output_path.relative_to(PROJECT_ROOT)),
        }
    )
    print(f"Validated and wrote {output_path.name}")

Validated and wrote silver_pbp_2023.parquet


Validated and wrote silver_pbp_2024.parquet


Validated and wrote silver_pbp_2025.parquet


Validated and wrote silver_pbp_2026.parquet


## 4. Review the output summary

Each partition passed its assertions and round-trip check before appearing below. An exact duplicate count greater than zero should be reviewed even though the duplicate rows were safely removed. The in-progress 2026 count will change as Bronze refreshes.

In [4]:
validation_summary = pd.DataFrame(validation_results)
assert len(validation_summary) == len(SEASONS)
assert validation_summary["silver_rows"].eq(
    validation_summary["bronze_rows"] - validation_summary["exact_duplicates_removed"]
).all()
validation_summary

,season,bronze_rows,exact_duplicates_removed,silver_rows,columns,games,output
0,2023,49665,0,49665,372,285,data/silver/pbp/silver_pbp_2023.parquet
1,2024,49492,0,49492,372,285,data/silver/pbp/silver_pbp_2024.parquet
2,2025,48771,0,48771,372,285,data/silver/pbp/silver_pbp_2025.parquet
3,2026,14031,0,14031,372,80,data/silver/pbp/silver_pbp_2026.parquet
